In [5]:
!pip install -q fastapi uvicorn nest-asyncio pandas numpy scikit-learn xgboost joblib

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
import joblib
import numpy as np
import pandas as pd

MODEL_PATHS = {
    "xgboost": "/content/drive/MyDrive/DATN/xgboost_smote_mfcm_model.pkl",
    "random_forest": "/content/drive/MyDrive/DATN/rf_smote_mfcm_model.pkl"
}

model_artifacts = {}

for model_name, model_path in MODEL_PATHS.items():
    if os.path.exists(model_path):
        model_artifacts[model_name] = joblib.load(model_path)
        print(f"Đã load model: {model_name}")
    else:
        print(f"Chưa tìm thấy model: {model_name}")
        print("Đường dẫn:", model_path)

print("\nCác model hiện có:", list(model_artifacts.keys()))

Đã load model: xgboost
Đã load model: random_forest

Các model hiện có: ['xgboost', 'random_forest']


In [8]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict, List, Any, Optional

# =============================
# 1. Khởi tạo API
# =============================

app = FastAPI(
    title="Fetal Health Prediction API",
    description="API dự đoán fetal_health bằng nhiều model: XGBoost, RandomForest + SMOTE + MFCM",
    version="2.0.0"
)


# =============================
# 2. Input schema
# =============================

class PredictRequest(BaseModel):
    model_name: str = "xgboost"
    features: Dict[str, float]


class BatchPredictRequest(BaseModel):
    model_name: str = "xgboost"
    data: List[Dict[str, float]]


# =============================
# 3. Hàm lấy model
# =============================

def get_artifact(model_name: str):
    model_name = model_name.lower().strip()

    if model_name not in model_artifacts:
        raise HTTPException(
            status_code=400,
            detail={
                "message": "Model không tồn tại hoặc chưa được load.",
                "model_name": model_name,
                "available_models": list(model_artifacts.keys())
            }
        )

    return model_artifacts[model_name]


# =============================
# 4. Hàm MFCM
# =============================

def make_weight_array(W, length):
    if length == 0:
        return np.array([])

    if isinstance(W, (int, float, np.integer, np.floating)):
        return np.full(length, float(W))

    W = np.asarray(W).reshape(-1)

    if W.size == 1:
        return np.full(length, float(W[0]))

    if W.size != length:
        return np.ones(length)

    return W.astype(float)


def compute_distances(
    DataCV,
    V,
    Binary_Col,
    Nominal_Col,
    Ordinal_Col,
    Interval_Col,
    W_b,
    W_n,
    W_o,
    W_i,
    p=2
):
    DataCV = np.asarray(DataCV, dtype=float)
    V = np.asarray(V, dtype=float)

    n_samples = DataCV.shape[0]
    n_clusters = V.shape[0]

    distances = np.zeros((n_samples, n_clusters))

    Binary_Col = list(Binary_Col)
    Nominal_Col = list(Nominal_Col)
    Ordinal_Col = list(Ordinal_Col)
    Interval_Col = list(Interval_Col)

    W_b_arr = make_weight_array(W_b, len(Binary_Col))
    W_n_arr = make_weight_array(W_n, len(Nominal_Col))
    W_o_arr = make_weight_array(W_o, len(Ordinal_Col))
    W_i_arr = make_weight_array(W_i, len(Interval_Col))

    for i in range(n_samples):
        for k in range(n_clusters):
            total = 0.0

            if len(Binary_Col) > 0:
                diff = DataCV[i, Binary_Col] != V[k, Binary_Col]
                total += np.sum(W_b_arr * diff.astype(float))

            if len(Nominal_Col) > 0:
                diff = DataCV[i, Nominal_Col] != V[k, Nominal_Col]
                total += np.sum(W_n_arr * diff.astype(float))

            if len(Ordinal_Col) > 0:
                diff = np.abs(DataCV[i, Ordinal_Col] - V[k, Ordinal_Col]) ** p
                total += np.sum(W_o_arr * diff)

            if len(Interval_Col) > 0:
                diff = np.abs(DataCV[i, Interval_Col] - V[k, Interval_Col]) ** p
                total += np.sum(W_i_arr * diff)

            distances[i, k] = total ** (1 / p)

    return distances


def predict_mfcm_membership(
    DataCV,
    V,
    Binary_Col,
    Nominal_Col,
    Ordinal_Col,
    Interval_Col,
    W_b,
    W_n,
    W_o,
    W_i,
    m=2,
    p=2
):
    distances = compute_distances(
        DataCV,
        V,
        Binary_Col,
        Nominal_Col,
        Ordinal_Col,
        Interval_Col,
        W_b,
        W_n,
        W_o,
        W_i,
        p
    )

    n = DataCV.shape[0]
    c = V.shape[0]

    U_pred = np.zeros((n, c))

    for i in range(n):
        if np.any(distances[i] == 0):
            zero_index = np.where(distances[i] == 0)[0]
            U_pred[i, zero_index] = 1
            continue

        for k in range(c):
            denom = np.sum((distances[i, k] / distances[i, :]) ** (2 / (m - 1)))
            U_pred[i, k] = 1 / denom

    return U_pred


# =============================
# 5. Chuẩn bị input theo từng model
# =============================

def prepare_input(df: pd.DataFrame, artifact: dict) -> pd.DataFrame:
    scaler = artifact["scaler"]

    V_final = artifact["V_final"]

    Binary_Col = artifact["Binary_Col"]
    Nominal_Col = artifact["Nominal_Col"]
    Ordinal_Col = artifact["Ordinal_Col"]
    Interval_Col = artifact["Interval_Col"]

    W_b = artifact["W_b"]
    W_n = artifact["W_n"]
    W_o = artifact["W_o"]
    W_i = artifact["W_i"]

    m = artifact["m"]
    p = artifact["p"]

    feature_columns_original = artifact["feature_columns_original"]
    feature_columns_final = artifact["feature_columns_final"]
    mfcm_cols = artifact["mfcm_cols"]

    missing_cols = [
        col for col in feature_columns_original
        if col not in df.columns
    ]

    if missing_cols:
        raise HTTPException(
            status_code=400,
            detail={
                "message": "Thiếu cột đầu vào",
                "missing_columns": missing_cols
            }
        )

    X_new = df[feature_columns_original].copy()

    try:
        X_new = X_new.astype(float)
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Dữ liệu đầu vào phải là số."
        )

    # Scale giống lúc train
    X_new_scaled = pd.DataFrame(
        scaler.transform(X_new),
        columns=feature_columns_original,
        index=X_new.index
    )

    # Tính membership MFCM
    U_new_mfcm = predict_mfcm_membership(
        DataCV=X_new_scaled.values,
        V=V_final,
        Binary_Col=Binary_Col,
        Nominal_Col=Nominal_Col,
        Ordinal_Col=Ordinal_Col,
        Interval_Col=Interval_Col,
        W_b=W_b,
        W_n=W_n,
        W_o=W_o,
        W_i=W_i,
        m=m,
        p=p
    )

    U_new_df = pd.DataFrame(
        U_new_mfcm,
        columns=mfcm_cols,
        index=X_new_scaled.index
    )

    # Ghép feature scale + membership MFCM
    X_new_final = pd.concat([X_new_scaled, U_new_df], axis=1)

    # Sắp xếp đúng thứ tự cột như lúc train
    X_new_final = X_new_final[feature_columns_final]

    return X_new_final


# =============================
# 6. Hàm predict theo model_name
# =============================

def predict_dataframe(df: pd.DataFrame, model_name: str):
    artifact = get_artifact(model_name)

    selected_model = artifact["model"]
    class_names = artifact.get(
        "class_names",
        {
            1: "Bình thường",
            2: "Nghi ngờ",
            3: "Bệnh lý"
        }
    )

    # Với XGBoost: label_offset = 1
    # Với RandomForest: label_offset = 0
    label_offset = artifact.get("label_offset", 0)

    X_new_final = prepare_input(df, artifact)

    y_pred_raw = selected_model.predict(X_new_final)

    y_pred_raw = np.asarray(y_pred_raw).astype(int)

    # Đổi nhãn nếu cần
    y_pred_final = y_pred_raw + label_offset

    results = []

    for i, pred in enumerate(y_pred_final):
        pred_int = int(pred)

        label = class_names.get(pred_int)
        if label is None:
            label = class_names.get(str(pred_int), "Không xác định")

        results.append({
            "index": int(i),
            "model_name": model_name,
            "prediction_raw": int(y_pred_raw[i]),
            "prediction": pred_int,
            "prediction_label": label
        })

    return results


# =============================
# 7. Endpoint API
# =============================

@app.get("/")
def home():
    return {
        "message": "API đang chạy",
        "available_models": list(model_artifacts.keys())
    }


@app.get("/models")
def get_models():
    return {
        "available_models": list(model_artifacts.keys()),
        "usage": {
            "xgboost": {
                "model_name": "xgboost",
                "note": "XGBoost trả nhãn 0,1,2 nên API cộng label_offset = 1"
            },
            "random_forest": {
                "model_name": "random_forest",
                "note": "RandomForest trả nhãn gốc 1,2,3 nên label_offset = 0"
            }
        }
    }


@app.get("/features")
def get_features(model_name: str = "xgboost"):
    artifact = get_artifact(model_name)

    return {
        "model_name": model_name,
        "required_features": artifact["feature_columns_original"],
        "total_features": len(artifact["feature_columns_original"])
    }


@app.post("/predict")
def predict_one(request: PredictRequest):
    df = pd.DataFrame([request.features])

    result = predict_dataframe(
        df=df,
        model_name=request.model_name
    )

    return {
        "status": "success",
        "model_name": request.model_name,
        "result": result[0]
    }


@app.post("/predict-batch")
def predict_batch(request: BatchPredictRequest):
    if len(request.data) == 0:
        raise HTTPException(
            status_code=400,
            detail="Danh sách data không được rỗng."
        )

    df = pd.DataFrame(request.data)

    results = predict_dataframe(
        df=df,
        model_name=request.model_name
    )

    return {
        "status": "success",
        "model_name": request.model_name,
        "total": len(results),
        "results": results
    }

In [9]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

Test api

In [10]:
import requests

response = requests.get("http://127.0.0.1:8000/")
response.json()

INFO:     Started server process [13794]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:57364 - "GET / HTTP/1.1" 200 OK


{'message': 'API đang chạy', 'available_models': ['xgboost', 'random_forest']}

In [11]:
response = requests.get("http://127.0.0.1:8000/features")
response.json()

INFO:     127.0.0.1:57378 - "GET /features HTTP/1.1" 200 OK


{'model_name': 'xgboost',
 'required_features': ['baseline value',
  'accelerations',
  'fetal_movement',
  'uterine_contractions',
  'light_decelerations',
  'severe_decelerations',
  'prolongued_decelerations',
  'abnormal_short_term_variability',
  'mean_value_of_short_term_variability',
  'percentage_of_time_with_abnormal_long_term_variability',
  'mean_value_of_long_term_variability',
  'histogram_width',
  'histogram_min',
  'histogram_max',
  'histogram_number_of_peaks',
  'histogram_number_of_zeroes',
  'histogram_mode',
  'histogram_mean',
  'histogram_median',
  'histogram_variance',
  'histogram_tendency'],
 'total_features': 21}

In [12]:
import requests

url = "http://127.0.0.1:8000/predict"

data = {
    "model_name": "xgboost",
    "features": {
        "baseline value": 120,
        "accelerations": 0.003,
        "fetal_movement": 0,
        "uterine_contractions": 0.004,
        "light_decelerations": 0,
        "severe_decelerations": 0,
        "prolongued_decelerations": 0,
        "abnormal_short_term_variability": 73,
        "mean_value_of_short_term_variability": 0.5,
        "percentage_of_time_with_abnormal_long_term_variability": 43,
        "mean_value_of_long_term_variability": 2.4,
        "histogram_width": 64,
        "histogram_min": 62,
        "histogram_max": 126,
        "histogram_number_of_peaks": 2,
        "histogram_number_of_zeroes": 0,
        "histogram_mode": 120,
        "histogram_mean": 137,
        "histogram_median": 121,
        "histogram_variance": 73,
        "histogram_tendency": 1
    }
}

response = requests.post(url, json=data)
response.json()

INFO:     127.0.0.1:57386 - "POST /predict HTTP/1.1" 200 OK


{'status': 'success',
 'model_name': 'xgboost',
 'result': {'index': 0,
  'model_name': 'xgboost',
  'prediction_raw': 0,
  'prediction': 1,
  'prediction_label': 'Bình thường'}}

In [13]:
import requests

url = "http://127.0.0.1:8000/predict"

data = {
    "model_name": "random_forest",
    "features": {
        "baseline value": 120,
        "accelerations": 0.003,
        "fetal_movement": 0,
        "uterine_contractions": 0.004,
        "light_decelerations": 0,
        "severe_decelerations": 0,
        "prolongued_decelerations": 0,
        "abnormal_short_term_variability": 73,
        "mean_value_of_short_term_variability": 0.5,
        "percentage_of_time_with_abnormal_long_term_variability": 43,
        "mean_value_of_long_term_variability": 2.4,
        "histogram_width": 64,
        "histogram_min": 62,
        "histogram_max": 126,
        "histogram_number_of_peaks": 2,
        "histogram_number_of_zeroes": 0,
        "histogram_mode": 120,
        "histogram_mean": 137,
        "histogram_median": 121,
        "histogram_variance": 73,
        "histogram_tendency": 1
    }
}

response = requests.post(url, json=data)
response.json()

INFO:     127.0.0.1:57394 - "POST /predict HTTP/1.1" 200 OK


{'status': 'success',
 'model_name': 'random_forest',
 'result': {'index': 0,
  'model_name': 'random_forest',
  'prediction_raw': 1,
  'prediction': 1,
  'prediction_label': 'Bình thường'}}

APP

In [14]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
!pip install -q gradio openpyxl xlsxwriter pytesseract opencv-python pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not handshake: Error in the pull function. [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out [IP: 185.125.190.80 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.


In [15]:
import os
import re
import cv2
import pandas as pd
import numpy as np
import gradio as gr
import pytesseract
from datetime import datetime

# ======================================================
# APP DỰ ĐOÁN TỪ EXCEL / CSV / ẢNH
# Yêu cầu: đã chạy cell load model_artifacts và các hàm get_artifact, predict_dataframe ở phía trên
# ======================================================

CLASS_NOTE = {
    1: "Bình thường",
    2: "Nghi ngờ",
    3: "Bệnh lý"
}

SAMPLE_VALUES = {
    "baseline value": 120,
    "accelerations": 0.003,
    "fetal_movement": 0,
    "uterine_contractions": 0.004,
    "light_decelerations": 0,
    "severe_decelerations": 0,
    "prolongued_decelerations": 0,
    "abnormal_short_term_variability": 73,
    "mean_value_of_short_term_variability": 0.5,
    "percentage_of_time_with_abnormal_long_term_variability": 43,
    "mean_value_of_long_term_variability": 2.4,
    "histogram_width": 64,
    "histogram_min": 62,
    "histogram_max": 126,
    "histogram_number_of_peaks": 2,
    "histogram_number_of_zeroes": 0,
    "histogram_mode": 120,
    "histogram_mean": 137,
    "histogram_median": 121,
    "histogram_variance": 73,
    "histogram_tendency": 1
}


def check_models_loaded():
    if "model_artifacts" not in globals() or len(model_artifacts) == 0:
        raise gr.Error(
            "Chưa load được model. Hãy chạy lại cell load model từ Google Drive trước khi mở app."
        )


def get_available_models():
    check_models_loaded()
    return list(model_artifacts.keys())


def get_required_features(model_name):
    check_models_loaded()
    artifact = get_artifact(model_name)
    return list(artifact["feature_columns_original"])


def get_required_features_text(model_name):
    cols = get_required_features(model_name)
    text = "File Excel hoặc dữ liệu trong ảnh cần có các trường sau, đúng thứ tự hoặc đúng tên trường:\n\n"
    text += "\n".join([f"{i+1}. {col}" for i, col in enumerate(cols)])
    return text


def make_template_file(model_name):
    cols = get_required_features(model_name)

    # Tạo 1 dòng dữ liệu mẫu để người dùng biết định dạng nhập
    template_df = pd.DataFrame([{col: SAMPLE_VALUES.get(col, 0) for col in cols}])

    output_path = f"/content/mau_file_du_doan_{model_name}.xlsx"
    template_df.to_excel(output_path, index=False)
    return output_path


# ======================================================
# PHẦN 1: ĐỌC FILE EXCEL / CSV
# ======================================================

def read_input_file(file_path):
    if file_path is None:
        raise gr.Error("Bạn cần tải lên file Excel hoặc CSV.")

    file_path = str(file_path)
    file_lower = file_path.lower()

    try:
        if file_lower.endswith(".csv"):
            df = pd.read_csv(file_path)
        elif file_lower.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            raise gr.Error("Chỉ hỗ trợ file .xlsx, .xls hoặc .csv.")
    except Exception as e:
        raise gr.Error(f"Không đọc được file. Lỗi: {e}")

    # Chuẩn hóa nhẹ tên cột để tránh lỗi thừa dấu cách đầu/cuối
    df.columns = [str(col).strip() for col in df.columns]
    return df


def validate_input_dataframe(df, model_name):
    required_cols = get_required_features(model_name)

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise gr.Error(
            "File hoặc ảnh đang thiếu các cột/trường sau:\n" + "\n".join(missing_cols)
        )

    # Chỉ lấy đúng cột đầu vào model, cột thừa vẫn được giữ lại trong file kết quả
    X_input = df[required_cols].copy()

    # Ép dữ liệu về số
    for col in required_cols:
        X_input[col] = pd.to_numeric(X_input[col], errors="coerce")

    # Kiểm tra giá trị bị trống hoặc không chuyển được sang số
    null_counts = X_input.isna().sum()
    bad_cols = null_counts[null_counts > 0]

    if len(bad_cols) > 0:
        msg = "Một số cột có giá trị trống hoặc không phải dạng số:\n"
        msg += "\n".join([f"- {col}: {int(count)} ô lỗi" for col, count in bad_cols.items()])
        msg += "\n\nHãy kiểm tra lại file Excel hoặc ảnh OCR trước khi dự đoán."
        raise gr.Error(msg)

    return X_input


def build_prediction_result(df_original, X_input, model_name, source_name="excel"):
    try:
        results = predict_dataframe(df=X_input, model_name=model_name)
    except Exception as e:
        raise gr.Error(f"Lỗi khi dự đoán: {e}")

    result_df = pd.DataFrame(results)

    # Bỏ các cột kỹ thuật không cần hiển thị nếu có
    drop_cols = ["index", "model_name", "prediction_raw"]
    result_display = result_df.drop(columns=drop_cols, errors="ignore")

    final_df = pd.concat(
        [df_original.reset_index(drop=True), result_display.reset_index(drop=True)],
        axis=1
    )

    # Đổi tên cột kết quả cho dễ đọc
    final_df = final_df.rename(columns={
        "prediction": "du_doan_fetal_health",
        "prediction_label": "ket_luan_du_doan"
    })

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"/content/ket_qua_du_doan_{source_name}_{model_name}_{timestamp}.xlsx"
    final_df.to_excel(output_path, index=False)

    summary = (
        f"Dự đoán thành công {len(final_df)} dòng bằng model: {model_name}.\n\n"
        "Ý nghĩa nhãn dự đoán:\n"
        "1 = Bình thường\n"
        "2 = Nghi ngờ\n"
        "3 = Bệnh lý"
    )

    return final_df, summary, output_path


def predict_from_excel(file_path, model_name):
    check_models_loaded()

    df_original = read_input_file(file_path)
    if df_original.empty:
        raise gr.Error("File không có dữ liệu.")

    X_input = validate_input_dataframe(df_original, model_name)
    return build_prediction_result(df_original, X_input, model_name, source_name="excel")


# ======================================================
# PHẦN 2: XỬ LÝ ẢNH OCR -> DATAFRAME -> DỰ ĐOÁN
# ======================================================

import difflib
from PIL import Image


def preprocess_image_for_ocr(image_path):
    """
    Tiền xử lý ảnh để OCR dễ đọc hơn.
    """
    image_path = str(image_path)
    image = cv2.imread(image_path)

    if image is None:
        raise gr.Error("Không đọc được ảnh. Hãy kiểm tra lại file ảnh.")

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Phóng to ảnh để OCR đọc chữ nhỏ tốt hơn
    gray = cv2.resize(
        gray,
        None,
        fx=2,
        fy=2,
        interpolation=cv2.INTER_CUBIC
    )

    # Giảm nhiễu nhẹ
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # Nhị phân hóa ảnh
    _, thresh = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return thresh


def ocr_image_to_text(image_path):
    """
    Đọc nội dung text từ ảnh bằng Tesseract OCR.
    """
    processed_img = preprocess_image_for_ocr(image_path)

    configs = [
        "--psm 6",
        "--psm 4",
        "--psm 11"
    ]

    texts = []
    for config in configs:
        try:
            text = pytesseract.image_to_string(
                processed_img,
                config=config
            )
            if text and text.strip():
                texts.append(text)
        except Exception:
            pass

    if len(texts) == 0:
        return ""

    # Ghép nhiều kết quả OCR để tăng khả năng bắt đủ trường
    return "\n".join(texts)


def clean_ocr_text(text):
    """
    Làm sạch text OCR trước khi tách key-value.
    """
    text = str(text)

    # Chuẩn hóa dấu ngoặc kép và dấu câu OCR hay đọc lệch
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("‘", "'")
    text = text.replace("’", "'")
    text = text.replace("：", ":")
    text = text.replace("；", ";")

    # Xóa ký tự lạ
    text = text.replace("™", "")
    text = text.replace("\x0c", " ")
    text = text.replace("\n", " ")

    # Sửa lỗi mất dấu gạch dưới trong tên trường
    text = text.replace("long term", "long_term")
    text = text.replace("short term", "short_term")
    text = text.replace("histogram width", "histogram_width")
    text = text.replace("histogram number", "histogram_number")
    text = text.replace("number of peaks", "number_of_peaks")
    text = text.replace("number of zeroes", "number_of_zeroes")
    text = text.replace("number_of peaks", "number_of_peaks")
    text = text.replace("number_of zeroes", "number_of_zeroes")

    # Gom nhiều khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_key(key):
    """
    Chuẩn hóa tên trường OCR về dạng có thể so sánh.
    """
    key = str(key).lower().strip()

    key = key.replace("long term", "long_term")
    key = key.replace("short term", "short_term")
    key = key.replace("histogram width", "histogram_width")
    key = key.replace("histogram number", "histogram_number")
    key = key.replace("number of peaks", "number_of_peaks")
    key = key.replace("number of zeroes", "number_of_zeroes")
    key = key.replace("number_of peaks", "number_of_peaks")
    key = key.replace("number_of zeroes", "number_of_zeroes")

    # Chỉ giữ chữ, số, dấu cách, dấu _
    key = re.sub(r"[^a-z0-9_ ]+", "", key)

    # Đưa khoảng trắng về _
    key = re.sub(r"\s+", "_", key)
    key = re.sub(r"_+", "_", key)
    key = key.strip("_")

    return key


def normalize_value(value):
    """
    Chuẩn hóa giá trị số OCR đọc được.
    Ví dụ:
    12@ -> 120
    @.0@3 -> 0.003
    @ -> 0
    0.003, -> 0.003
    """
    value = str(value).strip()

    # OCR hay đọc số 0 thành @, O, o
    value = value.replace("@", "0")
    value = value.replace("O", "0")
    value = value.replace("o", "0")
    value = value.replace(";", " ")
    value = value.replace(":", " ")

    # Lấy số đầu tiên, không ăn dấu phẩy cuối câu
    match = re.search(r"-?(?:\d+(?:[\.,]\d+)?|[\.,]\d+)", value)

    if not match:
        return None

    number_text = match.group(0).replace(",", ".")

    if number_text.startswith("."):
        number_text = "0" + number_text

    if number_text.startswith("-."):
        number_text = "-0" + number_text[1:]

    try:
        return float(number_text)
    except Exception:
        return None


def correct_common_ocr_value_by_column(col, value, raw_value):
    """
    Sửa một vài lỗi OCR rất thường gặp ở các cột dạng tỷ lệ nhỏ.
    Ví dụ:
    OCR đọc 0 thành 9 hoặc 6.
    OCR đọc 0.004 thành 9.004.
    """
    if value is None:
        return None

    low_rate_cols = {
        "accelerations",
        "fetal_movement",
        "uterine_contractions",
        "light_decelerations",
        "severe_decelerations",
        "prolongued_decelerations"
    }

    if col not in low_rate_cols:
        return value

    raw = str(raw_value).strip()
    raw = raw.replace("@", "0")
    raw = raw.replace("O", "0")
    raw = raw.replace("o", "0")
    raw = raw.replace(",", ".")

    # Trường hợp OCR đọc 0.004 thành 9.004 hoặc 6.004
    match = re.search(r"^[96]\.(\d+)", raw)
    if match:
        fixed_text = "0." + match.group(1)
        try:
            return float(fixed_text)
        except Exception:
            return value

    # Trường hợp OCR đọc 0 thành 9 hoặc 6 ở các cột tỷ lệ
    if value in [6, 9, 6.0, 9.0]:
        return 0.0

    return value


def extract_numbers_from_ocr_text(ocr_text):
    """
    Trường hợp ảnh chỉ có một dòng số, không có tên trường.
    Hàm này lấy toàn bộ số OCR đọc được.
    """
    text = str(ocr_text)

    text = text.replace("@", "0")
    text = text.replace("O", "0")
    text = text.replace("o", "0")
    text = text.replace(",", ".")

    numbers = re.findall(r"-?(?:\d+(?:\.\d+)?|\.\d+)", text)

    cleaned_numbers = []
    for number in numbers:
        if number.startswith("."):
            number = "0" + number
        elif number.startswith("-."):
            number = "-0" + number[1:]
        cleaned_numbers.append(float(number))

    return cleaned_numbers


def extract_features_from_ocr_text(ocr_text, required_cols):
    """
    Tách dữ liệu từ OCR text thành DataFrame đúng các trường model cần.
    Ưu tiên dạng:
    "tên trường": giá trị

    Nếu ảnh chỉ có số hoặc OCR key-value bị lỗi, sẽ thử gán số theo đúng thứ tự cột.
    """
    text = clean_ocr_text(ocr_text)

    expected_norm_map = {
        normalize_key(col): col for col in required_cols
    }

    result = {}

    # Bắt các cặp dạng:
    # "baseline value": 120
    # "histogram_max":; 126
    # "mean_value_of_short_term_variability": @.5
    pair_pattern = r'["\']?([A-Za-z0-9_ ]+?)["\']?\s*[:]\s*[;=\-–—]*\s*([@\dOo\.,\-]+)'
    pairs = re.findall(pair_pattern, text)

    for raw_key, raw_value in pairs:
        norm_key = normalize_key(raw_key)
        value = normalize_value(raw_value)

        if value is None:
            continue

        true_col = None

        # Khớp chính xác
        if norm_key in expected_norm_map:
            true_col = expected_norm_map[norm_key]

        else:
            # Khớp gần đúng khi OCR đọc sai nhẹ tên trường
            close_match = difflib.get_close_matches(
                norm_key,
                list(expected_norm_map.keys()),
                n=1,
                cutoff=0.68
            )

            if close_match:
                true_col = expected_norm_map[close_match[0]]

        if true_col is not None:
            value = correct_common_ocr_value_by_column(
                col=true_col,
                value=value,
                raw_value=raw_value
            )
            result[true_col] = value

    missing_cols = [col for col in required_cols if col not in result]

    # Nếu key-value vẫn thiếu, thử fallback bằng thứ tự số.
    # Cách này rất hữu ích khi OCR đọc tên cột bị lỗi nhưng vẫn đọc đủ 21 số.
    numbers = extract_numbers_from_ocr_text(text)

    if missing_cols and len(numbers) >= len(required_cols):
        numbers = numbers[:len(required_cols)]

        df = pd.DataFrame(
            [numbers],
            columns=required_cols
        )

        # Sửa lỗi OCR 6/9 -> 0 ở các cột dạng tỷ lệ nhỏ
        for col in required_cols:
            if col in df.columns:
                df.loc[0, col] = correct_common_ocr_value_by_column(
                    col=col,
                    value=df.loc[0, col],
                    raw_value=str(df.loc[0, col])
                )

        return df

    if missing_cols:
        raise gr.Error(
            "OCR chưa đọc đủ dữ liệu.\n\n"
            f"Đã đọc được: {len(result)}/{len(required_cols)} trường.\n\n"
            "Các trường còn thiếu:\n"
            + "\n".join(missing_cols)
            + "\n\nHãy dùng ảnh rõ hơn, crop sát vùng bảng, hoặc nhập bằng file Excel mẫu.\n\n"
            "Nội dung OCR đọc được:\n"
            + str(ocr_text)
        )

    df = pd.DataFrame(
        [[result[col] for col in required_cols]],
        columns=required_cols
    )

    return df


def predict_from_image(image_path, model_name):
    """
    Hàm dùng cho Gradio:
    ảnh -> OCR -> DataFrame -> validate -> predict -> xuất Excel

    Trả về đúng 4 outputs:
    1. Bảng kết quả
    2. Text OCR
    3. Thông báo
    4. File Excel kết quả
    """
    check_models_loaded()

    if image_path is None:
        raise gr.Error("Bạn cần tải lên ảnh chứa dữ liệu cần dự đoán.")

    required_cols = get_required_features(model_name)

    # 1. OCR ảnh
    ocr_text = ocr_image_to_text(image_path)

    if ocr_text is None or str(ocr_text).strip() == "":
        raise gr.Error("OCR không đọc được nội dung nào từ ảnh. Hãy dùng ảnh rõ hơn.")

    # 2. Tách dữ liệu từ OCR
    input_df = extract_features_from_ocr_text(
        ocr_text=ocr_text,
        required_cols=required_cols
    )

    # 3. Kiểm tra dữ liệu đầu vào
    X_input = validate_input_dataframe(
        df=input_df,
        model_name=model_name
    )

    # 4. Dự đoán và xuất file
    final_df, summary, output_path = build_prediction_result(
        df_original=input_df,
        X_input=X_input,
        model_name=model_name,
        source_name="image"
    )

    summary = (
        "Đã đọc ảnh và dự đoán thành công.\n\n"
        + summary
        + "\n\nLưu ý: Bạn nên kiểm tra lại bảng dữ liệu OCR trước khi dùng kết quả, "
        "vì OCR có thể đọc nhầm số nếu ảnh bị mờ hoặc chữ quá nhỏ."
    )

    return final_df, ocr_text, summary, output_path



# ======================================================
# GIAO DIỆN GRADIO
# ======================================================

available_models = get_available_models()
default_model = "xgboost" if "xgboost" in available_models else available_models[0]

with gr.Blocks(title="Fetal Health Prediction App") as demo:
    gr.Markdown("# App dự đoán sức khỏe thai nhi")
    gr.Markdown(
        "App hỗ trợ dự đoán từ **Excel/CSV** hoặc từ **ảnh chụp bảng dữ liệu**. "
        "Dữ liệu sẽ được xử lý theo pipeline đã train: scale dữ liệu, tính MFCM membership và dự đoán bằng model đã lưu."
    )

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=available_models,
            value=default_model,
            label="Chọn mô hình dự đoán"
        )
        template_button = gr.Button("Tải file Excel mẫu")

    required_box = gr.Textbox(
        value=get_required_features_text(default_model),
        label="Danh sách cột/trường bắt buộc",
        lines=12
    )

    template_file = gr.File(label="File mẫu")

    with gr.Tabs():
        with gr.Tab("Dự đoán từ Excel/CSV"):
            input_file = gr.File(
                label="Tải file Excel/CSV cần dự đoán",
                file_types=[".xlsx", ".xls", ".csv"],
                type="filepath"
            )

            predict_excel_button = gr.Button("Dự đoán từ Excel/CSV")

            excel_output_table = gr.Dataframe(label="Bảng kết quả dự đoán")
            excel_output_message = gr.Textbox(label="Thông báo", lines=6)
            excel_output_file = gr.File(label="Tải file kết quả Excel")

        with gr.Tab("Dự đoán từ ảnh"):
            gr.Markdown(
                "Ảnh nên là ảnh chụp rõ nét của bảng dữ liệu hoặc form dạng `tên trường: giá trị`. "
                "Nếu ảnh chỉ có một dòng số liệu, các số phải theo đúng thứ tự cột ở danh sách bên trên."
            )

            input_image = gr.Image(
                label="Tải ảnh chứa dữ liệu cần dự đoán",
                type="filepath"
            )

            predict_image_button = gr.Button("Đọc ảnh và dự đoán")

            image_output_table = gr.Dataframe(label="Bảng dữ liệu OCR + kết quả dự đoán")
            ocr_textbox = gr.Textbox(label="Nội dung OCR đọc được từ ảnh", lines=10)
            image_output_message = gr.Textbox(label="Thông báo", lines=10)
            image_output_file = gr.File(label="Tải file kết quả Excel")

    model_dropdown.change(
        fn=get_required_features_text,
        inputs=model_dropdown,
        outputs=required_box
    )

    template_button.click(
        fn=make_template_file,
        inputs=model_dropdown,
        outputs=template_file
    )

    predict_excel_button.click(
        fn=predict_from_excel,
        inputs=[input_file, model_dropdown],
        outputs=[excel_output_table, excel_output_message, excel_output_file]
    )

    predict_image_button.click(
        fn=predict_from_image,
        inputs=[input_image, model_dropdown],
        outputs=[image_output_table, ocr_textbox, image_output_message, image_output_file]
    )

# share=True để Colab tạo link public cho app
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92b35f7839bfdb4eea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
